# 📦 Tuần 19: Phân Cụm Dữ Liệu với K-Means

## Hôm nay học gì?

Hôm nay chúng ta học **K-Means Clustering** — một thuật toán học máy giúp **tự động nhóm dữ liệu** thành các cụm (cluster) dựa trên sự tương đồng.

---

## Vấn đề thực tế: Phân khúc khách hàng

Hãy tưởng tượng bạn là chủ một trung tâm thương mại với **200 khách hàng**.
Bạn muốn gửi email marketing nhưng không thể gửi cùng một nội dung cho tất cả:
- Khách giàu, chi tiêu nhiều → Giới thiệu hàng cao cấp
- Khách thu nhập thấp, chi tiêu ít → Gửi voucher giảm giá
- Khách giàu nhưng tiết kiệm → Thuyết phục bằng chương trình tích điểm

**Câu hỏi:** Làm sao tự động phân loại 200 khách hàng thành các nhóm tương đồng?

→ **K-Means Clustering** giải quyết bài toán này!

---

## So sánh với đời thường

K-Means giống như việc **sắp xếp kẹo vào hộp**:
- Bạn có một đống kẹo nhiều màu lộn xộn
- Bạn muốn nhóm chúng vào K hộp (K = số nhóm bạn muốn)
- Kẹo cùng màu (cùng tính chất) → vào cùng một hộp
- Mỗi hộp có một "kẹo đại diện" (centroid) ở giữa

---

## Tổng quan bài học

| Phần | Nội dung |
|------|----------|
| 1 | Khái niệm Clustering là gì? |
| 2 | Thuật toán K-Means hoạt động như thế nào? |
| 3 | Thực hành với dữ liệu khách hàng thật |
| 4 | Chuẩn hóa dữ liệu trước khi phân cụm |
| 5 | Chọn K tối ưu: Phương pháp Elbow & Silhouette |
| 6 | Ưu/nhược điểm và khi nào dùng K-Means |

In [ ]:
# ===== CELL 2: Import thư viện =====
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Cấu hình biểu đồ đẹp
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("Đã import xong tất cả thư viện cần thiết!")

---
## Phần 1: Clustering là gì?

### Học có giám sát vs Học không giám sát

Trước đây chúng ta học **Linear Regression, Decision Tree, Random Forest** — tất cả đều là **học có giám sát (Supervised Learning)**:
- Dữ liệu có **nhãn (label)** sẵn
- Ví dụ: ảnh mèo/chó đã được dán nhãn "mèo" hay "chó"
- Model học từ dữ liệu có nhãn → dự đoán nhãn cho dữ liệu mới

**Clustering** là **học không giám sát (Unsupervised Learning)**:
- Dữ liệu **KHÔNG có nhãn**
- Chúng ta không biết trước có bao nhiêu loại
- Model tự tìm cấu trúc ẩn trong dữ liệu

### So sánh trực quan:

| | Supervised Learning | Unsupervised (Clustering) |
|---|---|---|
| Dữ liệu | Có nhãn (label) | Không có nhãn |
| Mục tiêu | Dự đoán nhãn mới | Tìm nhóm tự nhiên |
| Ví dụ | Phân loại email spam | Phân khúc khách hàng |
| Tuần học | Week 17-18 | Week 19 (hôm nay!) |

### Ứng dụng của Clustering:
- **Marketing:** Phân khúc khách hàng để marketing có mục tiêu
- **Y tế:** Nhóm bệnh nhân theo triệu chứng tương tự
- **Netflix/Spotify:** Phân nhóm người dùng để gợi ý nội dung
- **Ảnh:** Nén ảnh bằng cách nhóm màu tương tự

In [ ]:
# ===== CELL: Minh họa trực quan Clustering =====
# Tạo dữ liệu giả để minh họa concept
np.random.seed(42)

# 3 cụm điểm dữ liệu
cluster1 = np.random.randn(30, 2) + [1, 1]   # Cụm 1: quanh điểm (1,1)
cluster2 = np.random.randn(30, 2) + [5, 5]   # Cụm 2: quanh điểm (5,5)
cluster3 = np.random.randn(30, 2) + [9, 1]   # Cụm 3: quanh điểm (9,1)

# Ghép tất cả lại thành một tập dữ liệu không có nhãn
all_points = np.vstack([cluster1, cluster2, cluster3])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ bên trái: Dữ liệu KHÔNG có nhãn (như thực tế)
axes[0].scatter(all_points[:, 0], all_points[:, 1], color='gray', alpha=0.6, s=80)
axes[0].set_title('Dữ liệu ban đầu (KHÔNG có nhãn)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Đặc trưng 1')
axes[0].set_ylabel('Đặc trưng 2')

# Biểu đồ bên phải: Sau khi Clustering tìm ra nhóm
colors = ['#E74C3C', '#3498DB', '#2ECC71']
for i, (cluster, color) in enumerate(zip([cluster1, cluster2, cluster3], colors)):
    axes[1].scatter(cluster[:, 0], cluster[:, 1], color=color, alpha=0.6, s=80, label=f'Nhóm {i+1}')
axes[1].set_title('Sau khi Clustering (đã phân nhóm)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Đặc trưng 1')
axes[1].set_ylabel('Đặc trưng 2')
axes[1].legend()

plt.suptitle('Mục tiêu của Clustering: Tìm nhóm tự nhiên trong dữ liệu', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Nhìn vào biểu đồ:")
print("- Bên trái: 90 điểm dữ liệu, không biết chúng thuộc nhóm nào")
print("- Bên phải: Clustering tự động phát hiện 3 nhóm tự nhiên!")

---
## Phần 2: Thuật toán K-Means hoạt động như thế nào?

### K-Means là gì?

**K** = số lượng cụm (cluster) mà bạn muốn tạo (bạn phải tự chỉ định trước)

**Means** = trung bình (mean) — mỗi cụm được đại diện bởi điểm trung bình gọi là **centroid** (tâm cụm)

### 4 bước của K-Means (dễ hiểu)

Tưởng tượng bạn có 10 người bạn đứng trong sân, muốn chia thành 3 nhóm:

**Bước 1 - Chọn điểm đại diện ngẫu nhiên:**
> Chọn ngẫu nhiên 3 người làm "trưởng nhóm" (centroid)

**Bước 2 - Gán mỗi người vào nhóm gần nhất:**
> Mỗi người tính khoảng cách đến 3 trưởng nhóm → đi về nhóm nào gần nhất

**Bước 3 - Cập nhật trưởng nhóm:**
> Trong mỗi nhóm, tính vị trí trung bình → đó là trưởng nhóm mới

**Bước 4 - Lặp lại bước 2 & 3 cho đến khi ổn định:**
> Nếu không ai đổi nhóm nữa → xong!

---

### Khoảng cách Euclidean (Euclidean Distance)

K-Means dùng công thức khoảng cách thông thường (như thước kẻ):

$$d = \sqrt{(x_1 - \mu_1)^2 + (x_2 - \mu_2)^2}$$

**Giải thích bằng lời:**
- $(x_1, x_2)$ = tọa độ điểm dữ liệu
- $(\mu_1, \mu_2)$ = tọa độ của centroid (tâm cụm)
- Bình phương hiệu → cộng lại → căn bậc hai = khoảng cách thẳng!

**Ví dụ:** Điểm A(3, 4) và centroid C(0, 0)
$$d = \sqrt{(3-0)^2 + (4-0)^2} = \sqrt{9+16} = \sqrt{25} = 5$$

---

### Hàm mục tiêu WCSS

K-Means tối thiểu hóa **WCSS (Within-Cluster Sum of Squares)** — tổng bình phương khoảng cách từ mỗi điểm đến tâm cụm của nó:

$$J = \sum_{k=1}^{K} \sum_{x \in C_k} ||x - \mu_k||^2$$

**Nói đơn giản:** WCSS càng nhỏ → các điểm trong cùng nhóm càng gần nhau → phân cụm càng tốt!

In [ ]:
# ===== CELL: Minh họa từng bước K-Means =====
# Dữ liệu 6 khách hàng từ slide PDF (Thu nhập và Điểm chi tiêu)
customers_simple = np.array([
    [15, 39],   # Khách A
    [16, 81],   # Khách B
    [17, 6],    # Khách C
    [18, 77],   # Khách D
    [18, 6],    # Khách E
    [19, 40],   # Khách F
])
names = ['A', 'B', 'C', 'D', 'E', 'F']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Bước 1: Khởi tạo centroids ngẫu nhiên ---
centroids_init = np.array([[15, 39], [18, 77]])  # Chọn A và D làm centroid ban đầu
axes[0].scatter(customers_simple[:, 0], customers_simple[:, 1], 
                color='gray', s=150, zorder=5)
for i, name in enumerate(names):
    axes[0].annotate(name, customers_simple[i], textcoords='offset points', 
                     xytext=(8, 5), fontsize=11)
axes[0].scatter(centroids_init[:, 0], centroids_init[:, 1], 
                color=['red', 'blue'], s=400, marker='*', zorder=6, label='Centroids ban đầu')
axes[0].set_title('Bước 1: Chọn centroids ngẫu nhiên', fontweight='bold')
axes[0].set_xlabel('Thu nhập (k$)')
axes[0].set_ylabel('Điểm chi tiêu')
axes[0].legend()

# --- Bước 2: Gán điểm vào cụm gần nhất ---
# Tính khoảng cách và gán nhóm
colors_assigned = []
cluster_labels = []
for point in customers_simple:
    d1 = np.sqrt(np.sum((point - centroids_init[0])**2))
    d2 = np.sqrt(np.sum((point - centroids_init[1])**2))
    if d1 <= d2:
        colors_assigned.append('red')
        cluster_labels.append(0)
    else:
        colors_assigned.append('blue')
        cluster_labels.append(1)

cluster_labels = np.array(cluster_labels)
for i, (point, color, name) in enumerate(zip(customers_simple, colors_assigned, names)):
    axes[1].scatter(point[0], point[1], color=color, s=150, zorder=5)
    axes[1].annotate(name, point, textcoords='offset points', xytext=(8, 5), fontsize=11)
axes[1].scatter(centroids_init[:, 0], centroids_init[:, 1], 
                color=['red', 'blue'], s=400, marker='*', zorder=6)
axes[1].set_title('Bước 2: Gán vào cụm gần nhất', fontweight='bold')
axes[1].set_xlabel('Thu nhập (k$)')
axes[1].set_ylabel('Điểm chi tiêu')

# --- Bước 3: Cập nhật centroid mới ---
new_centroid1 = customers_simple[cluster_labels == 0].mean(axis=0)
new_centroid2 = customers_simple[cluster_labels == 1].mean(axis=0)
new_centroids = np.array([new_centroid1, new_centroid2])

for i, (point, color, name) in enumerate(zip(customers_simple, colors_assigned, names)):
    axes[2].scatter(point[0], point[1], color=color, s=150, zorder=5)
    axes[2].annotate(name, point, textcoords='offset points', xytext=(8, 5), fontsize=11)
axes[2].scatter(centroids_init[:, 0], centroids_init[:, 1], 
                color=['red', 'blue'], s=200, marker='*', alpha=0.3, label='Centroid cũ')
axes[2].scatter(new_centroids[:, 0], new_centroids[:, 1], 
                color=['darkred', 'darkblue'], s=400, marker='X', zorder=6, label='Centroid mới')
axes[2].set_title('Bước 3: Cập nhật centroid (tính trung bình)', fontweight='bold')
axes[2].set_xlabel('Thu nhập (k$)')
axes[2].set_ylabel('Điểm chi tiêu')
axes[2].legend()

plt.suptitle('Minh họa 3 bước đầu tiên của K-Means (K=2)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Kết quả phân cụm vòng đầu:")
for i, (name, label) in enumerate(zip(names, cluster_labels)):
    cluster_name = 'Nhóm Đỏ (thu nhập thấp - chi tiêu thấp)' if label == 0 else 'Nhóm Xanh (thu nhập thấp - chi tiêu cao)'
    print(f"  Khách {name}: {cluster_name}")
print(f"\nCentroid mới: Nhóm 1 = {new_centroid1}, Nhóm 2 = {new_centroid2}")

---
## Phần 3: Thực hành với dữ liệu khách hàng thật

### Bộ dữ liệu Mall Customers

Chúng ta có dữ liệu **200 khách hàng** của một trung tâm thương mại:

| Cột | Ý nghĩa |
|-----|---------|
| CustomerID | Mã khách hàng |
| Genre | Giới tính (Male/Female) |
| Age | Tuổi |
| Annual Income (k$) | Thu nhập hàng năm (nghìn đô) |
| Spending Score (1-100) | Điểm chi tiêu (1=ít, 100=rất nhiều) |

**Câu hỏi của chủ siêu thị:** Có thể chia 200 khách hàng thành các nhóm tự nhiên dựa trên thu nhập và điểm chi tiêu không?

In [ ]:
# ===== CELL: Đọc và khám phá dữ liệu =====
df = pd.read_csv('Mall_Customers.csv')

print("=== TỔNG QUAN DỮ LIỆU ===")
print(f"Số hàng: {df.shape[0]}, Số cột: {df.shape[1]}")
print()
print("5 dòng đầu:")
print(df.head())
print()
print("Thống kê mô tả:")
print(df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].describe().round(1))
print()
print("Kiểm tra dữ liệu thiếu:")
print(df.isnull().sum())

In [ ]:
# ===== CELL: Khám phá phân phối dữ liệu =====
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Phân phối Thu nhập
axes[0].hist(df['Annual Income (k$)'], bins=20, color='#3498DB', edgecolor='white', alpha=0.8)
axes[0].set_title('Phân phối Thu nhập', fontweight='bold')
axes[0].set_xlabel('Thu nhập (k$)')
axes[0].set_ylabel('Số khách hàng')

# Phân phối Điểm chi tiêu
axes[1].hist(df['Spending Score (1-100)'], bins=20, color='#E74C3C', edgecolor='white', alpha=0.8)
axes[1].set_title('Phân phối Điểm chi tiêu', fontweight='bold')
axes[1].set_xlabel('Điểm chi tiêu (1-100)')
axes[1].set_ylabel('Số khách hàng')

# Scatter plot: Thu nhập vs Điểm chi tiêu
axes[2].scatter(df['Annual Income (k$)'], df['Spending Score (1-100)'], 
                alpha=0.6, color='gray', s=60)
axes[2].set_title('Thu nhập vs Điểm chi tiêu', fontweight='bold')
axes[2].set_xlabel('Thu nhập (k$)')
axes[2].set_ylabel('Điểm chi tiêu (1-100)')

plt.suptitle('Khám phá dữ liệu Mall Customers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Quan sát từ scatter plot:")
print("→ Nhìn bằng mắt thường, bạn thấy khoảng mấy nhóm tự nhiên?")
print("→ Gợi ý: Hãy nhìn vào biểu đồ bên phải và đếm các 'đám mây điểm'")

---
## Phần 4: Chuẩn hóa dữ liệu (StandardScaler)

### Tại sao cần chuẩn hóa?

Hãy nhìn vào hai đặc trưng của chúng ta:
- **Thu nhập:** từ 15 đến 137 (k$) — đơn vị NGHÌN ĐÔ
- **Điểm chi tiêu:** từ 1 đến 99 — đơn vị ĐIỂM (1-100)

**Vấn đề:** K-Means dùng khoảng cách Euclidean. Nếu thu nhập có thể biến động 100+ đơn vị, còn điểm chi tiêu chỉ biến động 1-99 đơn vị, thì **thu nhập sẽ "thống trị"** việc tính khoảng cách → phân cụm sẽ bị lệch!

### Ví dụ cụ thể:

Khách A(20k$, 80 điểm) và Khách B(100k$, 79 điểm):
- Khoảng cách thô: $\sqrt{(100-20)^2 + (79-80)^2} = \sqrt{6400+1} ≈ 80$ → Chủ yếu do thu nhập
- Sau chuẩn hóa: Thu nhập và điểm chi tiêu đóng góp **bình đẳng**

### StandardScaler làm gì?

$$z = \frac{x - \mu}{\sigma}$$

- $x$ = giá trị gốc
- $\mu$ = trung bình của cột đó
- $\sigma$ = độ lệch chuẩn của cột đó
- $z$ = giá trị sau chuẩn hóa (thường nằm trong khoảng -3 đến +3)

**Kết quả:** Tất cả đặc trưng đều có trung bình = 0 và độ lệch chuẩn = 1 → Bình đẳng với nhau!

In [ ]:
# ===== CELL: Chuẩn hóa dữ liệu =====
# Chọn 2 đặc trưng để phân cụm
features = ['Annual Income (k$)', 'Spending Score (1-100)']
X = df[features]

print("=== TRƯỚC KHI CHUẨN HÓA ===")
print(f"Thu nhập: min={X['Annual Income (k$)'].min()}, max={X['Annual Income (k$)'].max()}, mean={X['Annual Income (k$)'].mean():.1f}")
print(f"Điểm chi tiêu: min={X['Spending Score (1-100)'].min()}, max={X['Spending Score (1-100)'].max()}, mean={X['Spending Score (1-100)'].mean():.1f}")

# Chuẩn hóa
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\n=== SAU KHI CHUẨN HÓA ===")
print(f"Thu nhập: min={X_scaled[:,0].min():.2f}, max={X_scaled[:,0].max():.2f}, mean={X_scaled[:,0].mean():.4f}")
print(f"Điểm chi tiêu: min={X_scaled[:,1].min():.2f}, max={X_scaled[:,1].max():.2f}, mean={X_scaled[:,1].mean():.4f}")

# Trực quan hóa so sánh
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X['Annual Income (k$)'], X['Spending Score (1-100)'], 
                alpha=0.6, color='gray', s=60)
axes[0].set_title('Dữ liệu GỐC (trước chuẩn hóa)', fontweight='bold')
axes[0].set_xlabel('Thu nhập (k$)')
axes[0].set_ylabel('Điểm chi tiêu')

axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], 
                alpha=0.6, color='#3498DB', s=60)
axes[1].set_title('Dữ liệu SAU CHUẨN HÓA', fontweight='bold')
axes[1].set_xlabel('Thu nhập (chuẩn hóa)')
axes[1].set_ylabel('Điểm chi tiêu (chuẩn hóa)')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].axvline(x=0, color='red', linestyle='--', alpha=0.5)

plt.suptitle('So sánh dữ liệu trước và sau chuẩn hóa', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nNhận xét:")
print("- Hình dạng scatter plot KHÔNG đổi (vị trí tương đối giữ nguyên)")
print("- Chỉ thay đổi đơn vị: tất cả xoay quanh điểm (0,0) với scale bằng nhau")

---
## Phần 5: Chạy K-Means và Visualize kết quả

### Thử với K=5

Từ scatter plot, chúng ta thấy có vẻ khoảng **5 nhóm** tự nhiên. Hãy thử K=5 xem sao!

In [ ]:
# ===== CELL: Chạy K-Means với K=5 =====
k = 5
model = KMeans(n_clusters=k, random_state=42, n_init=10)
model.fit(X_scaled)

# Lấy kết quả
labels = model.labels_        # Nhãn cụm của từng điểm (0, 1, 2, 3, 4)
centroids = model.cluster_centers_  # Tọa độ tâm cụm (trong không gian đã chuẩn hóa)
inertia = model.inertia_      # Tổng WCSS

# Gán nhãn vào DataFrame gốc
df_result = X.copy()
df_result['Cluster'] = labels

print(f"=== KẾT QUẢ K-MEANS VỚI K={k} ===")
print(f"\nWCSS (Inertia): {inertia:.2f}")
print(f"\nSố khách hàng trong mỗi nhóm:")
cluster_counts = df_result['Cluster'].value_counts().sort_index()
for cluster, count in cluster_counts.items():
    print(f"  Nhóm {cluster}: {count} khách hàng")

print(f"\nĐặc điểm trung bình của từng nhóm:")
cluster_summary = df_result.groupby('Cluster').mean().round(1)
print(cluster_summary)

In [ ]:
# ===== CELL: Trực quan hóa kết quả phân cụm =====
# Chuyển đổi tâm cụm về không gian gốc để vẽ
centroids_original = scaler.inverse_transform(centroids)

# Màu sắc cho 5 nhóm
palette = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12', '#9B59B6']
cluster_names = ['Nhóm 0', 'Nhóm 1', 'Nhóm 2', 'Nhóm 3', 'Nhóm 4']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Biểu đồ 1: Không gian GỐC ---
for i in range(k):
    mask = df_result['Cluster'] == i
    axes[0].scatter(
        df_result.loc[mask, 'Annual Income (k$)'],
        df_result.loc[mask, 'Spending Score (1-100)'],
        color=palette[i], label=cluster_names[i], s=80, alpha=0.7
    )
# Vẽ tâm cụm
axes[0].scatter(centroids_original[:, 0], centroids_original[:, 1],
                color='black', s=400, marker='X', zorder=10, label='Tâm cụm')
axes[0].set_title(f'K-Means (K={k}) - Không gian GỐC', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Thu nhập (k$)', fontsize=11)
axes[0].set_ylabel('Điểm chi tiêu (1-100)', fontsize=11)
axes[0].legend(loc='upper left')

# --- Biểu đồ 2: Không gian ĐÃ CHUẨN HÓA ---
for i in range(k):
    mask = df_result['Cluster'] == i
    axes[1].scatter(
        X_scaled[mask, 0],
        X_scaled[mask, 1],
        color=palette[i], label=cluster_names[i], s=80, alpha=0.7
    )
axes[1].scatter(centroids[:, 0], centroids[:, 1],
                color='black', s=400, marker='X', zorder=10, label='Tâm cụm')
axes[1].set_title(f'K-Means (K={k}) - Không gian CHUẨN HÓA', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Thu nhập (chuẩn hóa)', fontsize=11)
axes[1].set_ylabel('Điểm chi tiêu (chuẩn hóa)', fontsize=11)
axes[1].legend(loc='upper left')

plt.suptitle('Phân cụm khách hàng Mall với K-Means', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Phân tích 5 nhóm khách hàng:")
print("  Nhóm Thu nhập cao - Chi tiêu cao: Khách VIP, cần chăm sóc đặc biệt")
print("  Nhóm Thu nhập cao - Chi tiêu thấp: Tiềm năng, cần khuyến khích chi tiêu")
print("  Nhóm Thu nhập thấp - Chi tiêu cao: Cẩn thận, có thể mất khách nếu không có khuyến mãi")
print("  Nhóm Thu nhập thấp - Chi tiêu thấp: Cần chương trình giá rẻ, voucher")
print("  Nhóm Thu nhập trung bình - Chi tiêu trung bình: Nhóm phổ thông nhất")

---
## Phần 6: Chọn K tối ưu

### Vấn đề: K bao nhiêu là đủ?

K-Means yêu cầu bạn **phải chỉ định K trước** — nhưng làm sao biết K nào tốt nhất?

Có 2 phương pháp phổ biến:

### Phương pháp 1: Elbow Method (Phương pháp khuỷu tay)

**Ý tưởng:**
- Chạy K-Means với K = 2, 3, 4, ..., 10
- Ghi lại WCSS (Inertia) cho mỗi K
- Vẽ đồ thị WCSS theo K
- Tìm "khuỷu tay" — điểm mà WCSS giảm chậm hẳn lại

**Giải thích bằng hình ảnh:** Giống như khuỷu tay người! Phần trên cánh tay (WCSS giảm nhanh), phần cẳng tay (WCSS giảm chậm). Điểm khuỷu = K tối ưu.

### Phương pháp 2: Silhouette Score

**Ý tưởng:** Đo **chất lượng phân cụm** — mỗi điểm dữ liệu có "hạnh phúc" không khi ở nhóm hiện tại?

$$s = \frac{b - a}{\max(a, b)}$$

Trong đó:
- $a$ = khoảng cách trung bình từ điểm đến **các điểm cùng cụm** (trong cụm)
- $b$ = khoảng cách trung bình từ điểm đến **cụm gần nhất khác** (giữa các cụm)

**Giải thích bằng lời:** Silhouette = "Tôi có gần những người cùng nhóm hơn người nhóm khác không?"

| Giá trị | Ý nghĩa |
|---------|----------|
| Gần +1 | Điểm ở đúng nhóm, rất tốt |
| Gần 0 | Điểm nằm trên ranh giới giữa 2 nhóm |
| Âm | Điểm có thể bị phân loại sai nhóm |

→ **K tối ưu = K có Silhouette Score CAO NHẤT**

In [ ]:
# ===== CELL: Elbow Method và Silhouette Score =====
list_k = range(2, 11)  # Thử K từ 2 đến 10
list_inertia = []
list_silhouette = []

# Chạy K-Means cho mỗi K
for k_val in list_k:
    km = KMeans(n_clusters=k_val, random_state=42, n_init=10)
    km.fit(X_scaled)
    list_inertia.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_)
    list_silhouette.append(sil)

# Vẽ 2 biểu đồ cạnh nhau
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Elbow Method ---
axes[0].plot(list(list_k), list_inertia, 'o-', color='#E74C3C', linewidth=2, markersize=8)
axes[0].axvline(x=5, color='gray', linestyle='--', alpha=0.7, label='K=5 (điểm khuỷu)')
axes[0].set_title('Elbow Method', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Số cụm K', fontsize=11)
axes[0].set_ylabel('WCSS (Inertia)', fontsize=11)
axes[0].legend()
axes[0].set_xticks(list(list_k))

# Highlight điểm K=5
best_k_idx = list(list_k).index(5)
axes[0].scatter([5], [list_inertia[best_k_idx]], 
                color='red', s=200, zorder=5)

# --- Silhouette Score ---
best_k_sil = list(list_k)[list_silhouette.index(max(list_silhouette))]
axes[1].plot(list(list_k), list_silhouette, 'o-', color='#3498DB', linewidth=2, markersize=8)
axes[1].axvline(x=best_k_sil, color='gray', linestyle='--', alpha=0.7, 
                label=f'K={best_k_sil} (điểm cao nhất)')
axes[1].set_title('Silhouette Score', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Số cụm K', fontsize=11)
axes[1].set_ylabel('Silhouette Score', fontsize=11)
axes[1].legend()
axes[1].set_xticks(list(list_k))

# Highlight điểm cao nhất
best_sil_idx = list_silhouette.index(max(list_silhouette))
axes[1].scatter([best_k_sil], [max(list_silhouette)], 
                color='blue', s=200, zorder=5)

plt.suptitle('Chọn K tối ưu: Elbow Method và Silhouette Score', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("=== KẾT QUẢ ===")
print(f"Elbow Method gợi ý: K = 5 (điểm khuỷu tay)")
print(f"Silhouette Score cao nhất: K = {best_k_sil} (score = {max(list_silhouette):.3f})")
print()
print("Bảng chi tiết Silhouette Score:")
for k_val, sil in zip(list_k, list_silhouette):
    marker = " ← CAO NHẤT" if k_val == best_k_sil else ""
    print(f"  K={k_val}: {sil:.3f}{marker}")

---
## Phần 7: Diễn giải kết quả - Ý nghĩa kinh doanh

Sau khi phân cụm, câu hỏi quan trọng nhất là: **"Các nhóm này có ý nghĩa gì?"**

Thuật toán chỉ tìm ra nhóm — bạn phải **diễn giải** chúng!

In [ ]:
# ===== CELL: Phân tích chi tiết từng cụm =====
# Chạy lại K-Means với K=5 (kết quả tốt nhất)
final_model = KMeans(n_clusters=5, random_state=42, n_init=10)
final_model.fit(X_scaled)
df['Cluster'] = final_model.labels_

# Tóm tắt từng cụm
cluster_summary = df.groupby('Cluster').agg({
    'Annual Income (k$)': 'mean',
    'Spending Score (1-100)': 'mean',
    'Age': 'mean',
    'CustomerID': 'count'
}).round(1)
cluster_summary.columns = ['Thu nhập TB (k$)', 'Điểm chi tiêu TB', 'Tuổi TB', 'Số KH']

print("=== PHÂN TÍCH CHI TIẾT 5 NHÓM KHÁCH HÀNG ===")
print(cluster_summary)
print()

# Đặt tên cho các nhóm dựa trên đặc điểm
segment_names = {}
for cluster in range(5):
    income = cluster_summary.loc[cluster, 'Thu nhập TB (k$)']
    spending = cluster_summary.loc[cluster, 'Điểm chi tiêu TB']
    
    if income > 70 and spending > 60:
        segment_names[cluster] = 'VIP - Giàu & Chi tiêu nhiều'
    elif income > 70 and spending < 40:
        segment_names[cluster] = 'Tiềm năng - Giàu nhưng tiết kiệm'
    elif income < 40 and spending > 60:
        segment_names[cluster] = 'Rủi ro - Nghèo nhưng chi nhiều'
    elif income < 40 and spending < 40:
        segment_names[cluster] = 'Phổ thông thấp - Thu nhập & Chi tiêu thấp'
    else:
        segment_names[cluster] = 'Trung bình - Thu nhập & Chi tiêu vừa'

print("=== ĐẶT TÊN PHÂN KHÚC ===")
for cluster, name in segment_names.items():
    count = cluster_summary.loc[cluster, 'Số KH']
    print(f"  Nhóm {cluster}: {name} ({int(count)} khách hàng)")

# Vẽ biểu đồ tổng hợp
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
palette = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12', '#9B59B6']

# Scatter plot phân cụm
for i in range(5):
    mask = df['Cluster'] == i
    label = f'Nhóm {i}: {segment_names.get(i, f"Nhóm {i}")}'
    axes[0].scatter(
        df.loc[mask, 'Annual Income (k$)'],
        df.loc[mask, 'Spending Score (1-100)'],
        color=palette[i], label=label, s=80, alpha=0.7
    )
axes[0].set_title('Phân khúc khách hàng (K=5)', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Thu nhập (k$)', fontsize=11)
axes[0].set_ylabel('Điểm chi tiêu (1-100)', fontsize=11)
axes[0].legend(loc='upper left', fontsize=8)

# Bar chart: Số lượng khách hàng mỗi nhóm
counts = [cluster_summary.loc[i, 'Số KH'] for i in range(5)]
short_names = [f'Nhóm {i}' for i in range(5)]
axes[1].bar(short_names, counts, color=palette, edgecolor='white', linewidth=1.5)
for i, count in enumerate(counts):
    axes[1].text(i, count + 1, str(int(count)), ha='center', fontweight='bold')
axes[1].set_title('Số lượng khách hàng mỗi nhóm', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Nhóm khách hàng')
axes[1].set_ylabel('Số khách hàng')

plt.tight_layout()
plt.show()

---
## Phần 8: Ưu điểm, Nhược điểm và Khi nào dùng K-Means

### Ưu điểm

| Ưu điểm | Giải thích |
|---------|------------|
| Đơn giản, dễ hiểu | Thuật toán chỉ có 4 bước rõ ràng |
| Nhanh | Chạy tốt với tập dữ liệu lớn |
| Luôn hội tụ | Không bao giờ bị "mắc kẹt" mãi mãi |
| Dễ visualize | 2 đặc trưng → scatter plot đẹp |

### Nhược điểm

| Nhược điểm | Giải thích | Giải pháp |
|-----------|------------|----------|
| Phải chỉ định K trước | Không tự tìm số cụm | Dùng Elbow + Silhouette |
| Nhạy với khởi tạo ngẫu nhiên | Kết quả có thể khác nhau mỗi lần | Dùng `random_state=42` |
| Nhạy với outlier | Điểm bất thường kéo lệch tâm cụm | Loại outlier trước |
| Giả định cụm hình tròn | Không tốt với cụm hình dạng phức tạp | Dùng DBSCAN thay thế |

### So sánh với các thuật toán khác:

| Tiêu chí | K-Means | Decision Tree (Tuần 17) | Random Forest (Tuần 18) |
|----------|---------|------------------------|------------------------|
| Loại | Unsupervised | Supervised | Supervised |
| Cần nhãn? | Không | Có | Có |
| Mục tiêu | Phân nhóm | Phân loại/Dự đoán | Phân loại/Dự đoán |
| Ưu điểm | Đơn giản | Dễ giải thích | Độ chính xác cao |

In [ ]:
# ===== CELL: Minh họa nhược điểm - Nhạy với outlier =====
np.random.seed(42)

# Dữ liệu 2 cụm bình thường
normal_data = np.vstack([
    np.random.randn(50, 2) + [0, 0],    # Cụm 1
    np.random.randn(50, 2) + [6, 6]     # Cụm 2
])

# Thêm outlier
outliers = np.array([[15, 0], [0, 15], [-5, -5]])
data_with_outliers = np.vstack([normal_data, outliers])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# K-Means KHÔNG có outlier
km_clean = KMeans(n_clusters=2, random_state=42, n_init=10)
km_clean.fit(normal_data)
for i in range(2):
    mask = km_clean.labels_ == i
    axes[0].scatter(normal_data[mask, 0], normal_data[mask, 1], 
                    s=60, alpha=0.7, label=f'Cụm {i}')
axes[0].scatter(km_clean.cluster_centers_[:, 0], km_clean.cluster_centers_[:, 1],
                color='black', marker='X', s=300, zorder=10, label='Tâm cụm')
axes[0].set_title('K-Means KHÔNG có outlier', fontweight='bold')
axes[0].set_xlabel('X1')
axes[0].set_ylabel('X2')
axes[0].legend()

# K-Means CÓ outlier
km_outlier = KMeans(n_clusters=2, random_state=42, n_init=10)
km_outlier.fit(data_with_outliers)
all_colors = plt.cm.Set1(np.linspace(0, 1, 2))
for i in range(2):
    mask = km_outlier.labels_ == i
    axes[1].scatter(data_with_outliers[mask, 0], data_with_outliers[mask, 1], 
                    s=60, alpha=0.7, label=f'Cụm {i}')
# Đánh dấu outlier
axes[1].scatter(outliers[:, 0], outliers[:, 1], 
                color='red', marker='*', s=400, zorder=10, label='Outlier', edgecolors='black')
axes[1].scatter(km_outlier.cluster_centers_[:, 0], km_outlier.cluster_centers_[:, 1],
                color='black', marker='X', s=300, zorder=10, label='Tâm cụm (bị lệch)')
axes[1].set_title('K-Means CÓ outlier (tâm cụm bị kéo lệch!)', fontweight='bold')
axes[1].set_xlabel('X1')
axes[1].set_ylabel('X2')
axes[1].legend()

plt.suptitle('Nhược điểm: K-Means nhạy cảm với Outlier', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Bài học:")
print("→ Luôn kiểm tra và xử lý outlier TRƯỚC khi chạy K-Means!")
print("→ Chuẩn hóa dữ liệu cũng giúp giảm ảnh hưởng của outlier")

---
## Tóm tắt & Ghi nhớ

### Những điểm quan trọng nhất tuần này

| # | Điểm chính | Chi tiết |
|---|------------|----------|
| 1 | **Clustering = Học không giám sát** | Dữ liệu không có nhãn, tự tìm nhóm tự nhiên |
| 2 | **K = Số cụm** phải chỉ định trước | Không tự tìm — cần Elbow/Silhouette để chọn |
| 3 | **4 bước K-Means** | Init → Assign → Update → Repeat |
| 4 | **Chuẩn hóa dữ liệu** là BẮT BUỘC | Dùng StandardScaler trước khi chạy K-Means |
| 5 | **Elbow Method** | Vẽ WCSS theo K, tìm điểm khuỷu |
| 6 | **Silhouette Score** | -1 đến +1, càng gần +1 càng tốt |
| 7 | **Diễn giải kết quả** | Thuật toán chỉ cho số — bạn phải đặt ý nghĩa |

### Quy trình phân cụm K-Means chuẩn:

```
1. Tải dữ liệu & khám phá (EDA)
       ↓
2. Chọn đặc trưng phù hợp
       ↓
3. Chuẩn hóa (StandardScaler)
       ↓
4. Chạy Elbow + Silhouette để chọn K
       ↓
5. Chạy KMeans với K tối ưu
       ↓
6. Visualize và diễn giải kết quả
       ↓
7. Rút ra insight kinh doanh
```

### So sánh với kiến thức các tuần trước:

| Tuần | Thuật toán | Loại | Dùng khi |
|------|-----------|------|----------|
| Week 16 | Linear Regression | Supervised | Dự đoán số liên tục |
| Week 17 | Decision Tree | Supervised | Phân loại/Dự đoán |
| Week 18 | Random Forest | Supervised | Phân loại chính xác hơn |
| **Week 19** | **K-Means** | **Unsupervised** | **Phân nhóm dữ liệu không có nhãn** |

---

### Câu hỏi thực hành:

1. Bạn đang làm việc cho một công ty điện thoại với dữ liệu cước gọi và mức chi tiêu của 1000 khách hàng. **Làm thế nào bạn áp dụng K-Means?** (Gợi ý: đặc trưng nào? K bằng mấy?)

2. Tại sao chúng ta **CẦN chuẩn hóa dữ liệu** trước K-Means? Điều gì xảy ra nếu không chuẩn hóa?

3. Elbow Method và Silhouette Score đôi khi cho K khác nhau. **Bạn nên chọn K nào?** Tại sao?

Hãy thử suy nghĩ và đặt câu hỏi nếu chưa rõ nhé! 💪